# KeelNet Stage 1 Experiment Template

Use this notebook as the Stage 1 working file for the official team workflow:

1. edit locally in VS Code
2. open this notebook in browser Google Colab
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive

This notebook contains the Stage 1 notes and the full A/B experiment flow in one place.


## Stage Notes

### Goal

Test whether explicit abstention training improves the trade-off between answer quality and unsupported answers on `SQuAD 2.0`.

This stage does **not** try to solve hallucination in general.

It tests one narrow claim:

> A model trained to abstain on unsupported questions should produce fewer unsupported answers than an answer-only baseline, without collapsing answer quality on answerable questions.

### Exact Experimental Question

Given one question and one evidence passage:

- does explicit abstention training reduce unsupported answers on unanswerable examples?
- how much answer quality is lost, if any, on answerable examples?

### Fixed Decisions

- dataset: `SQuAD 2.0` only
- input: one question and one passage
- output: answer span or `ABSTAIN`
- architecture family: one extractive QA backbone for both runs
- no retrieval
- no external verifier
- no custom scoring framework

### Recommended Default

Use a standard extractive QA setup with `distilbert-base-uncased` unless you have a clear reason to change it.

### Experimental Runs

- Run A: answer-only baseline trained only on answerable examples
- Run B: abstain-aware model trained on answerable and unanswerable examples

### Fair Comparison Rules

Keep these fixed across both runs:

- backbone
- tokenizer
- preprocessing
- max sequence length
- stride
- optimizer
- learning rate
- batch size
- number of epochs
- random seed if possible

### Evaluation Reminder

Use the official dev split for final comparison and check all three:

- unsupported-answer rate goes down
- answerable `F1` does not collapse
- the abstain-aware model is not simply over-abstaining

Useful companion files:

- `results-template.md`
- `../../docs/experiment-guidelines.md`


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in browser Google Colab. It mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the Stage 1 branch, and installs the project.

Path reminder:

- code runs from `/content/KeelNet`
- team artifacts should save to `/content/drive/MyDrive/KeelNet`
- share that `KeelNet` folder with your teammates in Google Drive
- if `DRIVE_PROJECT_DIR` does not start with `/content/drive/`, the path is wrong

Important:

- edit code locally in VS Code if you want
- execute this notebook in browser Colab, not a normal local Jupyter kernel
- push code to GitHub
- rerun this cell before training so the Colab runtime updates `/content/KeelNet`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


# Team default: save artifacts in a shared folder under MyDrive.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")

GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
GIT_BRANCH = "stage/01-grounded-abstention-baseline"
LOCAL_REPO_DIR = Path("/content/KeelNet")


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def mount_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    if not str(DRIVE_PROJECT_DIR).startswith("/content/drive/"):
        raise ValueError(
            f"DRIVE_PROJECT_DIR must point inside /content/drive, got: {DRIVE_PROJECT_DIR}"
        )
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {DRIVE_PROJECT_DIR}")


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def ensure_repo() -> Path:
    if (LOCAL_REPO_DIR / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=LOCAL_REPO_DIR)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(LOCAL_REPO_DIR)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    return LOCAL_REPO_DIR


mount_drive()
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime repo dir: {REPO_DIR}")
print(f"Using repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


<h2 style="color: #1d4ed8;">2. Configure The Run</h2>

Set `AUTHOR_NAME` to your name. This notebook builds the Stage 1 `RUN_NAME` automatically as `yourname-stage1-v1`, `yourname-stage1-v2`, `yourname-stage1-v3`, and so on based on completed runs.

This cell also prints the important values you should check before training.

It creates a small `RUN_STARTED.txt` file in the Drive run folder immediately, so you can confirm the Drive path is correct before training finishes.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

import pandas as pd
import torch

REPO_DIR = Path(REPO_DIR).resolve()

# Change only this for each teammate. The notebook builds the stage name and next version automatically.
AUTHOR_NAME = "yourname"
RUN_BASENAME = f"{AUTHOR_NAME}-stage1"
ARTIFACTS_ROOT = DRIVE_PROJECT_DIR / "artifacts" / "stage1_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"
STAGE_TITLE = "Stage 1: Grounded Abstention Baseline"
TARGET_METRICS = [
    "Run A answerable F1",
    "Run B answerable F1",
    "Run A unsupported-answer rate",
    "Run B unsupported-answer rate",
    "Run B abstain F1",
    "Run B selected threshold",
]


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
BASELINE_DIR = OUTPUT_ROOT / "baseline"
ABSTAIN_DIR = OUTPUT_ROOT / "abstain"
BASELINE_EVAL = OUTPUT_ROOT / "baseline_eval.json"
ABSTAIN_EVAL = OUTPUT_ROOT / "abstain_eval.json"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME
RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"repo_dir={REPO_DIR}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

# Keep these fixed across baseline and abstain runs for a fair comparison.
MODEL_NAME = "distilbert-base-uncased"
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

# Set these to small values for a quick smoke test before the full run.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

# Optional quick validation step before full training.
RUN_SMOKE_TEST = False
SMOKE_TEST_TRAIN_SAMPLES = 32
SMOKE_TEST_EVAL_SAMPLES = 32
SMOKE_TEST_EPOCHS = 1

print(f"Repo dir: {REPO_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def run(cmd):
    print("$", " ".join(str(part) for part in cmd))
    completed = subprocess.run([str(part) for part in cmd], cwd=REPO_DIR, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()


<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before training. This confirms the installed code path is at least minimally healthy inside the Colab runtime.


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])


<h2 style="color: #1d4ed8;">4. Optional Smoke Test</h2>

This cell runs a tiny baseline-only training command. Use it to catch path, dependency, and runtime issues before the full experiment.

Set `RUN_SMOKE_TEST = False` in the config cell if you want to skip it after the environment is already trusted.


In [ ]:
if RUN_SMOKE_TEST:
    smoke_dir = OUTPUT_ROOT / "smoke-baseline"
    run([
        sys.executable,
        "-m",
        "keelnet.train",
        "--mode",
        "baseline",
        "--output-dir",
        smoke_dir,
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(SMOKE_TEST_EPOCHS),
        "--train-batch-size",
        "2",
        "--eval-batch-size",
        "2",
        "--max-train-samples",
        str(SMOKE_TEST_TRAIN_SAMPLES),
        "--max-eval-samples",
        str(SMOKE_TEST_EVAL_SAMPLES),
    ])
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True in the config cell to run it.")


In [ ]:
print("DRIVE_PROJECT_DIR =", DRIVE_PROJECT_DIR)
print("ARTIFACTS_ROOT =", ARTIFACTS_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("exists DRIVE_PROJECT_DIR:", DRIVE_PROJECT_DIR.exists())
print("exists OUTPUT_ROOT:", OUTPUT_ROOT.exists())

for p in sorted(DRIVE_PROJECT_DIR.rglob("*")):
    print(p)


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 1 work area.
<ul>
  <li><strong>Start here:</strong> <code>src/keelnet/config.py</code>, <code>src/keelnet/data.py</code>, <code>src/keelnet/train.py</code>, <code>src/keelnet/evaluate.py</code>, <code>src/keelnet/postprocess.py</code>, <code>src/keelnet/metrics.py</code></li>
  <li><strong>Finish here:</strong> the answer-only baseline and abstain-aware run are comparable on one question plus one passage.</li>
  <li><strong>Out of scope:</strong> retrieval, support verification, calibration, confidence control, adaptive balancing.</li>
</ul>
</div>


## IMPLEMENTATION 1: Train The Two Stage 1 Runs

This is the main Stage 1 implementation and run section. Everything before this point is setup, sync, or validation.

Train the answer-only baseline and then the abstain-aware model. Keep the shared settings fixed so the comparison stays fair.


In [ ]:
def train_run(mode: str, output_dir: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.train",
        "--mode",
        mode,
        "--output-dir",
        str(output_dir),
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(NUM_EPOCHS),
        "--train-batch-size",
        str(TRAIN_BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_TRAIN_SAMPLES is not None:
        cmd.extend(["--max-train-samples", str(MAX_TRAIN_SAMPLES)])
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


train_run("baseline", BASELINE_DIR)
train_run("abstain", ABSTAIN_DIR)

## IMPLEMENTATION 2: Evaluate Both Runs

This cell evaluates both trained models and writes the result JSON files into your Drive run folder.


In [ ]:
def evaluate_run(mode: str, model_dir: Path, output_path: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.evaluate",
        "--mode",
        mode,
        "--model-path",
        str(model_dir),
        "--output-path",
        str(output_path),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


evaluate_run("baseline", BASELINE_DIR, BASELINE_EVAL)
evaluate_run("abstain", ABSTAIN_DIR, ABSTAIN_EVAL)


## IMPLEMENTATION 3: Compare The Main Metrics

This cell loads the saved evaluation JSON files and builds the main comparison table for Stage 1.


In [ ]:
baseline_results = json.loads(BASELINE_EVAL.read_text())
abstain_results = json.loads(ABSTAIN_EVAL.read_text())

comparison = pd.DataFrame(
    [
        {"model": "Run A", **baseline_results["dev_metrics"]},
        {"model": "Run B", **abstain_results["dev_metrics"]},
    ]
)
comparison = comparison[
    [
        "model",
        "answerable_em",
        "answerable_f1",
        "unsupported_answer_rate",
        "abstain_precision",
        "abstain_recall",
        "abstain_f1",
        "overall_em",
        "overall_f1",
    ]
]
comparison = comparison.rename(
    columns={
        "overall_em": "decision_aware_overall_em",
        "overall_f1": "decision_aware_overall_f1",
    }
)
comparison.round(2)


## IMPLEMENTATION 4: Inspect The Abstain Threshold

This cell shows the validation-selected threshold and validation metrics for the abstain-aware run.


In [ ]:
print("Selected abstain threshold:", abstain_results["selected_threshold"])
print("\nValidation metrics for Run B:")
pd.Series(abstain_results["validation_metrics"]).round(2)


## IMPLEMENTATION 5: Plot The Threshold Sweep

This cell visualizes the abstain-aware trade-off across thresholds on the dev split and marks the selected threshold.


In [ ]:
import matplotlib.pyplot as plt

sweep = pd.DataFrame(abstain_results["threshold_sweep"]["dev"])
ax = sweep.plot(
    x="threshold",
    y=["answerable_f1", "unsupported_answer_rate", "abstain_f1"],
    marker="o",
    figsize=(8, 4.5),
    grid=True,
)
ax.axvline(
    abstain_results["selected_threshold"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="selected threshold",
)
ax.set_title("Run B threshold sweep on dev split")
ax.set_xlabel("Null-score difference threshold")
ax.set_ylabel("Metric value")
ax.legend()
plt.tight_layout()
plt.show()


## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


## Save Notes And Review Artifacts

This cell creates a teammate-friendly note file inside the Drive run folder and lists the current artifacts. Update the generated note as you learn what does and does not work in Stage 1.


In [ ]:
if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                "",
                "## Goal",
                "- One-sentence objective: compare the answer-only baseline against the abstain-aware run on the same grounded QA setup",
                "- Success condition: unsupported-answer rate goes down without answerable F1 collapsing",
                "- Out of scope: retrieval, support verification, calibration, and unsupported-confidence penalties",
                "",
                "## Commands",
                "- Smoke test command(s): optional small baseline training run",
                "- Main command(s): train baseline, train abstain, evaluate both, inspect threshold sweep",
                "- Input artifacts or checkpoints: baseline and abstain model directories under OUTPUT_ROOT",
                "- Output files to inspect: baseline_eval.json, abstain_eval.json, threshold sweep plot",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- unsupported but confident answer",
                "- answerable but over-abstained",
                "- wrong span despite supporting evidence",
                "- ambiguous question",
                "- context truncation problem",
                "- thresholding problem",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "git_branch": CURRENT_BRANCH,
            "output_root": str(OUTPUT_ROOT),
            "target_metrics": TARGET_METRICS,
            "baseline_eval": str(BASELINE_EVAL),
            "abstain_eval": str(ABSTAIN_EVAL),
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


<h2 style="color: #15803d;">Final Check</h2>

A win is not just a lower unsupported-answer rate.

Check all three:

- unsupported-answer rate goes down
- answerable F1 does not collapse
- the abstain-aware model is not simply over-abstaining

After that, inspect failure cases using the saved `*_eval.json` files under the printed `OUTPUT_ROOT` directory in Drive.


<h2 style="color: #15803d;">Share This Run</h2>

This cell prints a minimal share-ready summary for teammates, saves it into the Drive run folder, and marks the run as completed so the next run becomes the next version.

Use this after the evaluation cells finish.


In [ ]:
from datetime import datetime, timezone

baseline_summary = json.loads(BASELINE_EVAL.read_text(encoding="utf-8"))["dev_metrics"]
abstain_summary = json.loads(ABSTAIN_EVAL.read_text(encoding="utf-8"))["dev_metrics"]
share_lines = [
    "# Stage 1 Share Note",
    "",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    f"- Run A answerable_f1: {baseline_summary['answerable_f1']:.2f}",
    f"- Run B answerable_f1: {abstain_summary['answerable_f1']:.2f}",
    f"- Run A unsupported_answer_rate: {baseline_summary['unsupported_answer_rate']:.2f}",
    f"- Run B unsupported_answer_rate: {abstain_summary['unsupported_answer_rate']:.2f}",
    f"- Run B abstain_f1: {abstain_summary['abstain_f1']:.2f}",
    f"- Drive folder path: {OUTPUT_ROOT}",
]
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
print(f"Saved share note: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
